# MACE MLIP fine-tuning example

This notebook documents `../examples/mace_mlip_training_cpu_minimal.sh` as it is written now. The example launches `mace.cli.run_train.py` with the exact keywords shown in the script.

The command is a fine-tuning example: it uses `--foundation_model`, `--multiheads_finetuning`, `--pt_train_file`, `--device='cuda'`, and `--distributed`. It is not the earlier local CPU smoke test.

In [ ]:
from pathlib import Path

script_path = Path("../examples/mace_mlip_training_cpu_minimal.sh")
print(script_path.resolve())
print(script_path.read_text())

## Command shape

Run the script from the repository root after activating the Python environment used for MACE:

```bash
bash examples/mace_mlip_training_cpu_minimal.sh
```

The script currently contains absolute paths for `--foundation_model`, `--train_file`, and `--test_file`. Change those values in the script when running in a different location. Keep `--energy_key='ref_energy'` and `--forces_key='ref_forces'` aligned with the labels in the XYZ files.

## Data keywords

| Keyword | Value in the example | Meaning |
|---|---|---|
| `--train_file` | `/shared/home/training_omat/stratified_sample.xyz` | XYZ file used for training. |
| `--valid_fraction` | `0.2` | Fraction of `--train_file` used for validation. |
| `--test_file` | `/shared/home/training_omat/test.xyz` | XYZ file used for testing. |
| `--E0s` | `{8:-2049.91807262, 78: -518568.86288573, 22: -23328.13360429}` | Reference values used by the run. |
| `--energy_key` | `ref_energy` | Energy label read from the XYZ file. |
| `--forces_key` | `ref_forces` | Forces label read from the XYZ file. |

## Model keywords

| Keyword | Value in the example | Meaning |
|---|---|---|
| `--name` | `fine_tuning` | Run name used for outputs and restart files. |
| `--num_channels` | `128` | Channel count used by the model. |
| `--foundation_model` | `/shared/home/mace-omat-0-medium.model` | Model file used as the starting point. |
| `--multiheads_finetuning` | `True` | Enables multihead fine-tuning. |
| `--lr` | `0.01` | Learning rate. |
| `--scaling` | `rms_forces_scaling` | Scaling choice used by the run. |
| `--pt_train_file` | `omat` | Replay training source used by the run. |
| `--num_samples_pt` | `30000` | Number of replay samples used by the run. |

## Loss and batch keywords

| Keyword | Value in the example | Meaning |
|---|---|---|
| `--forces_weight` | `100` | Weight applied to the forces term. |
| `--energy_weight` | `1` | Weight applied to the energy term. |
| `--batch_size` | `4` | Training batch size. |
| `--valid_batch_size` | `4` | Validation batch size. |
| `--max_num_epochs` | `200` | Maximum number of epochs. |

## SWA, EMA, and optimiser keywords

| Keyword | Value in the example | Meaning |
|---|---|---|
| `--start_swa` | `300` | Epoch used to start SWA. |
| `--swa` | enabled | Enables SWA. |
| `--swa_forces_weight` | `100` | Forces weight used with SWA. |
| `--ema` | enabled | Enables EMA. |
| `--ema_decay` | `0.99` | EMA decay value. |
| `--amsgrad` | enabled | Enables AMSGrad. |
| `--error_table` | `PerAtomMAE` | Error table printed by the run. |

## Runtime keywords

| Keyword | Value in the example | Meaning |
|---|---|---|
| `--device` | `cuda` | Device used by the run. |
| `--distributed` | enabled | Enables distributed execution. |
| `--default_dtype` | `float64` | Default floating-point dtype. |
| `--seed` | `123` | Random seed. |
| `--restart_latest` | enabled | Restarts from the latest matching checkpoint. |
| `--save_cpu` | enabled | Saves CPU-loadable output. |

In [ ]:
import re
from pathlib import Path

script_text = Path("../examples/mace_mlip_training_cpu_minimal.sh").read_text()
keywords = re.findall(r"--[A-Za-z0-9_]+", script_text)
for keyword in dict.fromkeys(keywords):
    print(keyword)

## Practical checks

- Confirm that `--foundation_model`, `--train_file`, and `--test_file` point to files available from the machine running the script.
- Confirm that the structures in `--train_file` and `--test_file` contain `ref_energy` and `ref_forces` because the script sets `--energy_key='ref_energy'` and `--forces_key='ref_forces'`.
- Confirm that `--device='cuda'` and `--distributed` match the runtime you intend to use.
- Keep `--name` unchanged when you want `--restart_latest` to continue the same run; change `--name` when you want a separate run.